# Zapdos Cascade Demo — Motion-Gated YOLOv8n

**Goal:** measure how much cheaper continuous CCTV inference gets when
you put a `cv2.absdiff` motion gate in front of YOLOv8n.

**How to run:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. `Runtime -> Manage sessions` — confirm the GPU is attached.
3. Add your Roboflow API key to Colab Secrets as `ROBOFLOW_API_KEY`
   (left sidebar -> key icon -> Add new secret).
4. `Runtime -> Run all`. End-to-end runtime is ~10-15 minutes on a T4.

**What the notebook does, cell by cell:**
1. Install pinned deps
2. Clone the repo so we can import `src/motion_gate.py`, `src/detector.py`, `src/cascade.py`
3. Download the Roboflow Construction Site Safety v28 dataset
4. Synthesize a 1500-frame (5-min @ 5 fps) CCTV clip: 85% static, 15% active
5. Load YOLOv8n and warm up the GPU
6. Run baseline (detector on every frame)
7. Sweep motion thresholds to pick a good one
8. Run cascade with the chosen threshold
9. Convert timings to `$/camera/month`
10. Spot-check detector recall on labeled test images
11. Save all numbers to `results/summary.csv`

## 1. Install dependencies

In [1]:
# Pinned versions live in requirements.txt. Reinstalling in Colab is
# ~30-60s. `-q` keeps the log short.
!pip install -q ultralytics>=8.2.0 roboflow>=1.1.0 opencv-python-headless>=4.9.0 numpy>=1.24.0 pandas>=2.0.0 pyyaml>=6.0

## 2. Get the source files

The core logic lives in `src/motion_gate.py`, `src/detector.py`, `src/cascade.py`.
We clone the repo so this notebook stays a thin harness — the mechanism
is versioned in git, not buried in notebook cells.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Akshaj11/AI-Safety.git'
# The demo lives on this feature branch, not on main.
BRANCH = 'claude/zapdos-cascade-demo-7dgrib'
REPO_DIR = '/content/AI-Safety'
PROJECT_DIR = os.path.join(REPO_DIR, 'zapdos-cascade-demo')

# Wipe any stale clone so we always get the latest branch state.
!rm -rf $REPO_DIR
# --depth 1 keeps the clone small; -b picks the branch that has the demo.
!git clone --depth 1 -b $BRANCH $REPO_URL $REPO_DIR

assert os.path.isdir(PROJECT_DIR), f'{PROJECT_DIR} missing — check the branch name.'

# Add the project dir to sys.path so `from src.cascade import ...` works.
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Files:', sorted(os.listdir('.')))


## 3. Imports

In [ ]:
import time
import random
import glob
import numpy as np
import cv2
import pandas as pd
from pathlib import Path

# Our own modules (from the cloned repo).
from src.motion_gate import motion_score
from src.detector import Detector
from src.cascade import run_baseline, run_cascade, cost_per_camera_month

# Deterministic run so the '15% active' splice pattern is reproducible.
random.seed(42)
np.random.seed(42)
print('Imports OK')

## 4. Download the Roboflow dataset

We use **Construction Site Safety v28** (CC BY 4.0, 2,801 images).
The download is ~200 MB and takes 1-2 minutes on Colab's network.

**Setup:** in Colab, click the 🔑 icon in the left sidebar, add a secret
named `ROBOFLOW_API_KEY`, and toggle 'Notebook access' on.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

# Colab Secrets keeps the key out of the notebook JSON — safe to commit.
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY in Colab Secrets first.'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
# yolov8 format matches what YOLO expects if we ever want to fine-tune.
dataset = project.version(28).download('yolov8')

DATA_ROOT = Path(dataset.location)
print('Dataset at:', DATA_ROOT)
print('Splits:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

## 5. Build the synthetic streaming clip

Real labeled continuous CCTV is either behind research agreements
(VIRAT) or unlabeled (YouTube). For a *cost* measurement that's fine —
the GPU spends the same time on any frame regardless of provenance.

We build 1,500 frames = 5 minutes @ 5 fps:
- **85% static frames** = the same background image + tiny per-frame
  noise. This is what an empty warehouse aisle actually looks like.
- **15% active frames** = a random dataset image spliced in. This is
  what motion looks like to the gate.

Each frame gets a `tag` ('static' or 'active') so we can debug the
gate's decisions later.

In [ ]:
TOTAL_FRAMES = 1500       # 5 min at 5 fps
ACTIVE_RATIO = 0.15       # 15% of frames have real motion
FRAME_SHAPE = (480, 640)  # HxW — standard CCTV

# Grab all training-split JPEGs to sample from.
train_images = sorted(glob.glob(str(DATA_ROOT / 'train' / 'images' / '*.jpg')))
assert train_images, 'No training images found — check the dataset path.'
print(f'Have {len(train_images)} images to draw from')

def load_and_resize(path):
    img = cv2.imread(path)
    return cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

# One 'background' scene the static frames vary around.
background = load_and_resize(train_images[0])

frames = []  # list of (tag, frame) tuples — this is what the harness eats
for i in range(TOTAL_FRAMES):
    if random.random() < ACTIVE_RATIO:
        # Splice a random dataset image in as the 'active' frame.
        active = load_and_resize(random.choice(train_images))
        frames.append(('active', active))
    else:
        # Static frame = background + a tiny bit of noise. The noise
        # is intentional: perfectly identical frames would give the
        # gate a suspiciously easy win. Real CCTV has sensor noise,
        # flicker, and compression artifacts even on 'empty' scenes.
        noise = np.random.randint(-2, 3, background.shape, dtype=np.int16)
        static = np.clip(background.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        frames.append(('static', static))

n_active = sum(1 for tag, _ in frames if tag == 'active')
print(f'Built {len(frames)} frames: {n_active} active ({n_active/len(frames):.0%})')

## 6. Load YOLOv8n and warm up the GPU

First inference call on a fresh GPU includes CUDA init + kernel
compilation (~2-5 seconds). If we don't warm up, those seconds get
charged to whichever run happens first and skew the comparison.

In [ ]:
detector = Detector(weights_path='yolov8n.pt', confidence=0.25)

# Warm-up: 3 real inference calls so CUDA kernels are compiled and
# cudnn has picked its algorithm before we start timing.
for _ in range(3):
    detector.detect(frames[0][1])
print('Detector ready. Classes:', len(detector.names))

## 7. Baseline: YOLOv8n on every frame

This is the naive deployment. Every frame gets full inference.

In [ ]:
baseline = run_baseline(frames, detector)
print('Baseline result:')
for k, v in baseline.items():
    print(f'  {k}: {v}')

## 8. Pick a motion threshold

The threshold determines what counts as 'motion' vs 'static'. Too low
and the gate lets everything through (no savings). Too high and it
misses real activity (recall drops).

We sweep candidate thresholds, print the pass rate for each, then pick
the highest threshold that still lets ~all active frames through.

In [ ]:
# Score every consecutive pair once; reuse across thresholds.
scores = []
prev = None
for tag, frame in frames:
    scores.append((tag, motion_score(frame, prev)))
    prev = frame

print(f'{'threshold':>10} {'pass_rate':>10} {'active_kept':>12} {'static_passed':>14}')
for thr in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    n_pass = sum(1 for _, s in scores if s > thr)
    active_kept = sum(1 for tag, s in scores if tag == 'active' and s > thr)
    static_passed = sum(1 for tag, s in scores if tag == 'static' and s > thr)
    total_active = sum(1 for tag, _ in scores if tag == 'active')
    print(f'{thr:>10.1f} {n_pass/len(scores):>10.2%} {active_kept}/{total_active:>10} {static_passed:>14}')

# Pick the threshold: highest value that still keeps every 'active'
# frame. Bumping above that starts throwing away real motion.
# Adjust manually if the sweep suggests a different sweet spot.
MOTION_THRESHOLD = 2.0
print(f'\nChosen MOTION_THRESHOLD = {MOTION_THRESHOLD}')

## 9. Cascade: motion gate → YOLOv8n

Same frames, same detector. Only frames whose motion score exceeds
the threshold reach YOLOv8n.

In [ ]:
cascade = run_cascade(frames, detector, motion_threshold=MOTION_THRESHOLD)
print('Cascade result:')
for k, v in cascade.items():
    print(f'  {k}: {v}')

## 10. Convert timings to $/camera/month

Cost basis: AWS g4dn.xlarge, 1x T4, $0.526/hr on-demand, continuous
24/7 at 5 fps. See `src/cascade.py::cost_per_camera_month` for the
formula.

In [ ]:
FPS = 5
GPU_HOURLY = 0.526

cost_baseline = cost_per_camera_month(baseline['ms_per_frame'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)
cost_cascade = cost_per_camera_month(cascade['ms_per_frame_avg'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)

speedup = baseline['ms_per_frame'] / cascade['ms_per_frame_avg']
cost_reduction = cost_baseline / cost_cascade

print(f'Baseline: {baseline["ms_per_frame"]:.2f} ms/frame  |  ${cost_baseline:.2f}/camera/month')
print(f'Cascade:  {cascade["ms_per_frame_avg"]:.2f} ms/frame  |  ${cost_cascade:.2f}/camera/month')
print(f'Speedup:  {speedup:.2f}x')
print(f'Cost reduction: {cost_reduction:.2f}x')

## 11. Spot-check detector recall on labeled test images

The synthetic clip is fine for cost. To sanity-check that the gate
isn't hiding recall problems on *real* labeled data, we run both
configurations over the test split and count how many labeled images
still trigger a detection.

Caveat: stock YOLOv8n only knows COCO classes (person, car, ...). It
will NOT fire on 'NO-Hardhat' as a distinct class — for that you'd
fine-tune. So we only score classes YOLO already knows.

In [ ]:
test_images = sorted(glob.glob(str(DATA_ROOT / 'test' / 'images' / '*.jpg')))[:100]
print(f'Recall check on {len(test_images)} labeled test images')

hits_baseline = 0
hits_cascade = 0
prev = None

for path in test_images:
    img = cv2.imread(path)
    img = cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

    # Baseline: always run detector.
    r_base = detector.detect(img)
    if len(r_base.boxes) > 0:
        hits_baseline += 1

    # Cascade: only run detector if gate passes.
    if motion_score(img, prev) > MOTION_THRESHOLD:
        r_cas = detector.detect(img)
        if len(r_cas.boxes) > 0:
            hits_cascade += 1
    prev = img

recall_baseline = hits_baseline / len(test_images)
recall_cascade = hits_cascade / len(test_images)
print(f'Baseline recall: {recall_baseline:.2%} ({hits_baseline}/{len(test_images)})')
print(f'Cascade recall:  {recall_cascade:.2%} ({hits_cascade}/{len(test_images)})')
print(f'Recall delta:    {(recall_cascade - recall_baseline):+.2%}')

## 12. Save results to `results/summary.csv`

This is the file Ganesh sees when he browses the repo without opening
the notebook. Every assumption goes in the CSV so it's obvious what
the numbers are conditional on.

In [ ]:
import csv

os.makedirs('results', exist_ok=True)
rows = [
    ('metric', 'value'),
    ('baseline_ms_per_frame', round(baseline['ms_per_frame'], 2)),
    ('cascade_ms_per_frame', round(cascade['ms_per_frame_avg'], 2)),
    ('baseline_total_seconds', round(baseline['wall_time_s'], 2)),
    ('cascade_total_seconds', round(cascade['wall_time_s'], 2)),
    ('speedup', round(speedup, 2)),
    ('motion_threshold', MOTION_THRESHOLD),
    ('frames_through_detector_baseline', baseline['frames_processed']),
    ('frames_through_detector_cascade', cascade['frames_through_detector']),
    ('cost_baseline_per_camera_month', round(cost_baseline, 2)),
    ('cost_cascade_per_camera_month', round(cost_cascade, 2)),
    ('cost_reduction_x', round(cost_reduction, 2)),
    ('total_frames_in_clip', TOTAL_FRAMES),
    ('fps_assumed', FPS),
    ('gpu_hourly_usd', GPU_HOURLY),
    ('static_ratio_assumed', 1 - ACTIVE_RATIO),
]

with open('results/summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerows(rows)

print('Saved results/summary.csv:')
print(open('results/summary.csv').read())

# Download button: uncomment to grab the file to your local machine.
# from google.colab import files
# files.download('results/summary.csv')

## 13. Headline

Paste these into the README and writeup. Done.

In [ ]:
print('=' * 60)
print('HEADLINE — copy into README.md and writeup.pdf')
print('=' * 60)
print(f'Baseline:       {baseline["ms_per_frame"]:6.2f} ms/frame   ${cost_baseline:7.2f}/camera/month')
print(f'Cascade:        {cascade["ms_per_frame_avg"]:6.2f} ms/frame   ${cost_cascade:7.2f}/camera/month')
print(f'Cost reduction: {cost_reduction:.2f}x')
print(f'Motion threshold used: {MOTION_THRESHOLD}')
print(f'Gate pass rate:        {cascade["gate_pass_rate"]:.2%}')
print('=' * 60)